# Psychometric Item Embedding

**Logic:** Generate semantic embeddings for every unique item text via either a
local Ollama model or a hosted API (OpenAI / Google).

**Workflow:** `item_list` parquet -> embed -> item-keyed embedding parquet.

Runs once for each data split (training / holdout / validation).

In [1]:
import os
import sys

import numpy as np
import polars as pl

# Import the embedding-backend functions from the modular helper.
sys.path.append("r_functions")
from embedding_functions import get_embeddings, get_embeddings_API, get_embeddings_HF

## Configure splits

Which splits to embed: `""` (training), `"holdout_"` or `"validation_"`.

In [2]:
splits = ["", "holdout_", "validation_"]

## Embed each split

For every split: read the item list, generate embeddings, then stack them into a
wide item-keyed matrix and write it to parquet.

In [4]:
for split in splits:

    # ---- Read in Data ----

    item_list = (
        pl.read_parquet(f"../../data/raw/{split}item_list.parquet")
        .unique(subset="item", keep="first", maintain_order=True)
    )

    # ---- Generate Embeddings ----

    # Local Ollama model (qwen3-embedding:8b + task-specific instruction works best
    # for inter-item correlation prediction in my benchmarks)
    # embeddings, model_safe = get_embeddings(item_list)

    # API alternative: OpenAI text-embedding-3-large or Gemini text-embedding-005
    # embeddings, model_safe = get_embeddings_API(item_list, model="gemini-embedding-2", dims=3072)

    # --- HF models ---
    # "dwulff/mpnet-personality", "nvidia/NV-Embed-v2", "Alibaba-NLP/gte-Qwen2-7B-instruct", "intfloat/e5-mistral-7b-instruct"
    embeddings, model_safe = get_embeddings_HF(item_list, model="Alibaba-NLP/gte-Qwen2-7B-instruct", batch_size=64, quantize=4)


    # ---- Save Data ----
    # Stack embedding vectors into a wide matrix, attach the item text, prefix
    # columns with "emb" for downstream feature selectors.
    emb_matrix = np.vstack(list(embeddings.values()) if isinstance(embeddings, dict) else embeddings)
    item_embeddings_df = pl.from_numpy(
        emb_matrix,
        schema=[f"emb{i + 1}" for i in range(emb_matrix.shape[1])],
        orient="row",
    )
    item_embeddings_df.insert_column(0, item_list.get_column("item"))

    os.makedirs(f"../../data/raw/{model_safe}", exist_ok=True)
    item_embeddings_df.write_parquet(
        f"../../data/raw/{model_safe}/{split}embeddings_raw.parquet"
    )

`low_cpu_mem_usage` was None, now default to True since model is quantized.
Batches:   2%|▏         | 1/49 [01:51<1:28:54, 111.13s/it]


KeyboardInterrupt: 